# 04 — OOD Detection, Clustering & Ghost Class Discovery

This notebook implements the full Task 2 pipeline:

1. Build an unlabeled deployment pool mixing known-class test patches with ghost-class patches
2. Evaluate three OOD detection methods: Softmax Confidence, Energy Score, Mahalanobis Distance
3. Compare methods and select the best one
4. Analyze intermediate vs. final-layer features for OOD separation
5. Cluster OOD-flagged patches with UMAP + HDBSCAN
6. Name discovered ghost classes using visual and statistical evidence

Ground-truth labels are used **only** for final evaluation visualization — never for detection or clustering decisions.

## 0. Environment Setup (Colab / Local)

Run this cell first. It auto-detects Colab vs local Jupyter and sets up the environment.

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if IN_COLAB:
    REPO_URL = 'https://github.com/sabyasachi1992/eurosat-land-cover-ood.git'
    REPO_DIR = '/content/eurosat-land-cover-ood'
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
    if not os.path.isdir('EuroSAT/2750'):
        print('Downloading EuroSAT RGB dataset...')
        !wget --no-check-certificate -q https://madm.dfki.de/files/sentinel/EuroSAT.zip -O EuroSAT.zip
        import zipfile
        if not zipfile.is_zipfile('EuroSAT.zip'):
            print('DFKI mirror failed, trying Kaggle mirror...')
            !rm -f EuroSAT.zip
            !pip install -q kagglehub
            import kagglehub, glob
            path = kagglehub.dataset_download('apollo2506/eurosat-dataset')
            !mkdir -p EuroSAT
            candidates = glob.glob(f'{path}/**/2750', recursive=True)
            if candidates:
                !cp -r {candidates[0]} EuroSAT/2750
            else:
                !cp -r {path}/* EuroSAT/
        else:
            !unzip -q EuroSAT.zip
            !mkdir -p EuroSAT
            if os.path.isdir('2750'):
                !mv 2750 EuroSAT/2750
            elif os.path.isdir('EuroSAT_RGB'):
                !mv EuroSAT_RGB EuroSAT/2750
            !rm -f EuroSAT.zip
        print(f'Done. Classes: {sorted(os.listdir("EuroSAT/2750"))}')
    PROJECT_ROOT = REPO_DIR
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        PROJECT_ROOT = os.path.abspath('..')
    elif os.path.isfile('config.yaml'):
        PROJECT_ROOT = os.getcwd()
    else:
        PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')
assert os.path.isfile('config.yaml'), f'config.yaml not found in {os.getcwd()}'

## 1. Setup

In [ ]:
%matplotlib inline

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.config import Config
from src.utils.seed import set_global_seed
from src.data.dataset import discover_images, stratified_split, EuroSATDataset
from src.data.transforms import load_norm_stats, get_eval_transform
from src.data.pool import build_unlabeled_pool
from src.models.factory import create_model
from src.ood.energy import compute_energy_scores
from src.ood.mahalanobis import fit_class_gaussians, compute_mahalanobis_scores
from src.ood.features import FeatureExtractor
from src.evaluation.ood_metrics import compute_auroc, compute_fpr_at_tpr
from src.clustering.reducer import reduce_umap
from src.clustering.clusterer import cluster_hdbscan, compute_cluster_stats
from src.utils.visualization import (
    plot_ood_histograms, plot_umap_clusters, plot_patch_grid,
)

In [ ]:
# Load configuration and set global seed
config = Config.load('config.yaml')
set_global_seed(config.seed)

dataset_root = config.dataset_root
weights_path = config.weights_path
norm_stats_path = config.norm_stats_path
output_dir = config.output_dir

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Seed:   {config.seed}')

In [ ]:
# Load best model
model = create_model(config)
checkpoint = torch.load(weights_path, map_location=device)
assert checkpoint['architecture'] == config.architecture
model.load_state_dict(checkpoint['state_dict'])
model.to(device)
model.eval()

print(f'Loaded {config.architecture} (epoch {checkpoint["best_epoch"]}, val loss {checkpoint["best_val_loss"]:.4f})')

# Load normalization stats and build eval transform
mean, std = load_norm_stats(norm_stats_path)
eval_transform = get_eval_transform(mean, std)

## 2. Unlabeled Pool Construction

We build a shuffled pool that simulates post-deployment conditions: a mix of known-class patches
(that the model was trained on) and ghost-class patches (that the model has never seen).
Labels are stripped — ground truth is retained separately for evaluation only.

In [ ]:
# Reproduce the stratified split to get test paths
known_paths, known_labels = discover_images(dataset_root, config.known_classes)
train_paths, val_paths, test_paths, train_labels, val_labels, test_labels = stratified_split(
    known_paths, known_labels,
    train_ratio=config.train_ratio,
    val_ratio=config.val_ratio,
    test_ratio=config.test_ratio,
    seed=config.seed,
)

# Build unlabeled pool
pool_paths, pool_gt_labels = build_unlabeled_pool(
    known_test_paths=test_paths,
    known_test_labels=test_labels,
    ghost_root=dataset_root,
    ghost_classes=config.ghost_classes,
    n_known_samples=config.unlabeled_known_count,
    seed=config.seed,
)

pool_gt = np.array(pool_gt_labels)
n_known_in_pool = int((pool_gt == 0).sum())
n_ghost_in_pool = int((pool_gt == 1).sum())
print(f'Pool size: {len(pool_paths)} total')
print(f'  Known (ID):  {n_known_in_pool}')
print(f'  Ghost (OOD): {n_ghost_in_pool}')

In [ ]:
# Build DataLoader for the pool
pool_dataset = EuroSATDataset(
    file_paths=pool_paths,
    labels=pool_gt_labels,  # labels carried for DataLoader but NOT used in detection
    transform=eval_transform,
)

pool_loader = DataLoader(
    pool_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=0,
)

print(f'Pool DataLoader: {len(pool_loader)} batches')

## 3. Softmax Confidence Baseline

As a first approach, we use the maximum softmax probability as an OOD score.
The intuition is that the model should be less confident on OOD inputs.
We will see that this intuition is **wrong** for standard neural networks.

In [ ]:
# Run classifier on pool, get max softmax probabilities
all_softmax_probs = []

with torch.no_grad():
    for images, _ in pool_loader:
        images = images.to(device)
        logits = model(images)
        probs = F.softmax(logits, dim=1)
        max_probs = probs.max(dim=1).values
        all_softmax_probs.append(max_probs.cpu())

softmax_conf = torch.cat(all_softmax_probs).numpy()

# For OOD detection: lower confidence = more OOD
# We negate so that higher score = more OOD (convention for AUROC)
softmax_ood_scores = -softmax_conf

print(f'Softmax confidence — mean: {softmax_conf.mean():.4f}, std: {softmax_conf.std():.4f}')

In [ ]:
# Plot histogram of softmax confidence for known vs ghost
id_mask = pool_gt == 0
ood_mask = pool_gt == 1

plot_ood_histograms(
    id_scores=softmax_conf[id_mask],
    ood_scores=softmax_conf[ood_mask],
    method_name='Softmax Confidence (raw)',
    save_path=f'{output_dir}/figures/softmax_histogram.png',
)

In [ ]:
# Compute AUROC and FPR@95TPR
softmax_auroc = compute_auroc(softmax_ood_scores, pool_gt)
softmax_fpr95 = compute_fpr_at_tpr(softmax_ood_scores, pool_gt, tpr_target=0.95)

print(f'Softmax Confidence:')
print(f'  AUROC:      {softmax_auroc:.4f}')
print(f'  FPR@95TPR:  {softmax_fpr95:.4f}')

### Why Softmax Confidence Fails

The softmax confidence baseline performs poorly for OOD detection because:

1. **Softmax normalizes to sum=1**: The softmax function forces all output probabilities to sum to 1.0,
   regardless of the input. Even for completely random or OOD inputs, the model must distribute its
   confidence across the 6 known classes — there is no "I don't know" option.

2. **Neural networks are trained to be confident**: During training with cross-entropy loss, the model
   is rewarded for pushing the correct class probability toward 1.0. This makes the model inherently
   overconfident, even on inputs far from the training distribution.

3. **ReLU networks produce high-confidence extrapolations**: ReLU-based networks are piecewise linear.
   As inputs move away from the training manifold, logits can grow unboundedly, producing arbitrarily
   high softmax confidence on OOD samples.

4. **No calibration guarantee**: Standard training provides no mechanism to calibrate confidence with
   actual correctness probability. The model has never seen ghost-class patches, so it has no basis
   for assigning low confidence to them.

This motivates the need for dedicated OOD detection methods that go beyond raw softmax outputs.

## 4. Energy Score

The Energy Score (Liu et al., 2020) uses the log-sum-exp of logits as an OOD indicator.
Unlike softmax confidence, it uses the **full logit vector** rather than just the maximum,
providing better separation between ID and OOD samples.

Energy = −log(Σ exp(logit_k)). Lower energy → in-distribution, higher energy → OOD.

In [ ]:
# Compute energy scores
energy_scores = compute_energy_scores(model, pool_loader, device)

print(f'Energy scores — mean: {energy_scores.mean():.4f}, std: {energy_scores.std():.4f}')
print(f'  ID mean:  {energy_scores[id_mask].mean():.4f}')
print(f'  OOD mean: {energy_scores[ood_mask].mean():.4f}')

In [ ]:
# Plot histogram
plot_ood_histograms(
    id_scores=energy_scores[id_mask],
    ood_scores=energy_scores[ood_mask],
    method_name='Energy Score',
    save_path=f'{output_dir}/figures/energy_histogram.png',
)

In [ ]:
# Compute AUROC and FPR@95TPR
energy_auroc = compute_auroc(energy_scores, pool_gt)
energy_fpr95 = compute_fpr_at_tpr(energy_scores, pool_gt, tpr_target=0.95)

print(f'Energy Score:')
print(f'  AUROC:      {energy_auroc:.4f}')
print(f'  FPR@95TPR:  {energy_fpr95:.4f}')

## 5. Mahalanobis Distance

The Mahalanobis distance (Lee et al., 2018) operates in **feature space** rather than output space.
We fit class-conditional Gaussians on training features from an intermediate layer, then measure
how far each pool sample's features are from the nearest class distribution.

Higher Mahalanobis distance → more OOD.

In [ ]:
# Build training DataLoader for fitting Gaussians
train_dataset = EuroSATDataset(
    file_paths=train_paths,
    labels=train_labels,
    transform=eval_transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=0,
)

# Fit class-conditional Gaussians on training features
print(f'Fitting class-conditional Gaussians on layer: {config.ood_feature_layer}')
class_means, shared_cov = fit_class_gaussians(
    model, train_loader, config.ood_feature_layer, device
)
print(f'Fitted {len(class_means)} class Gaussians, feature dim: {shared_cov.shape[0]}')

In [ ]:
# Compute Mahalanobis scores on pool
mahal_scores = compute_mahalanobis_scores(
    model, pool_loader, config.ood_feature_layer,
    class_means, shared_cov, device
)

print(f'Mahalanobis scores — mean: {mahal_scores.mean():.4f}, std: {mahal_scores.std():.4f}')
print(f'  ID mean:  {mahal_scores[id_mask].mean():.4f}')
print(f'  OOD mean: {mahal_scores[ood_mask].mean():.4f}')

In [ ]:
# Plot histogram
plot_ood_histograms(
    id_scores=mahal_scores[id_mask],
    ood_scores=mahal_scores[ood_mask],
    method_name='Mahalanobis Distance',
    save_path=f'{output_dir}/figures/mahalanobis_histogram.png',
)

In [ ]:
# Compute AUROC and FPR@95TPR
mahal_auroc = compute_auroc(mahal_scores, pool_gt)
mahal_fpr95 = compute_fpr_at_tpr(mahal_scores, pool_gt, tpr_target=0.95)

print(f'Mahalanobis Distance:')
print(f'  AUROC:      {mahal_auroc:.4f}')
print(f'  FPR@95TPR:  {mahal_fpr95:.4f}')

## 6. OOD Method Comparison

We compare all three methods side-by-side to identify the best OOD detector.

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    'Method': ['Softmax Confidence', 'Energy Score', 'Mahalanobis Distance'],
    'AUROC': [softmax_auroc, energy_auroc, mahal_auroc],
    'FPR@95TPR': [softmax_fpr95, energy_fpr95, mahal_fpr95],
})

# Format for display
comparison['AUROC'] = comparison['AUROC'].map('{:.4f}'.format)
comparison['FPR@95TPR'] = comparison['FPR@95TPR'].map('{:.4f}'.format)
print(comparison.to_string(index=False))

### Method Justification

**Best method selection**: The method with the highest AUROC and lowest FPR@95TPR is preferred.

- **Softmax Confidence** serves as a weak baseline — it fails because softmax normalization
  forces confident predictions even on OOD inputs (as discussed above).

- **Energy Score** improves over softmax by using the full logit vector via log-sum-exp.
  It captures the overall magnitude of the model's response, not just the winning class.

- **Mahalanobis Distance** operates in feature space, measuring distributional distance rather
  than output confidence. It leverages the geometric structure of learned representations,
  which often provides the best OOD separation when the feature space is well-structured.

### Threshold Tradeoff

Choosing an OOD detection threshold involves an explicit tradeoff:

- **Low threshold** (aggressive): Catches more true OOD samples (high TPR) but also flags many
  known-class patches as OOD (high FPR). This means more patches are sent for manual review,
  increasing operational cost.

- **High threshold** (conservative): Fewer false alarms (low FPR) but misses more OOD samples
  (lower TPR). Ghost-class patches slip through undetected, which is dangerous if the goal is
  to discover all new terrain types.

The FPR@95TPR metric captures this tradeoff at a practical operating point: "if we want to catch
95% of OOD samples, how many known samples do we incorrectly flag?" Lower is better.

## 7. Feature Extraction Analysis

### Theoretical Argument: Why Final-Layer Features Are Poor for OOD Clustering

The final classification layer (the linear head) maps features into a 6-dimensional space
optimized to separate the 6 known classes. This creates several problems for OOD detection:

1. **Collapse into decision regions**: Training with cross-entropy loss encourages features to
   collapse into 6 tight clusters — one per known class. OOD samples, having no dedicated region,
   get mapped to whichever known-class cluster is nearest in this compressed space.

2. **Lost diversity**: The 4 ghost classes (HerbaceousVegetation, Pasture, PermanentCrop, River)
   have genuinely different visual characteristics, but the final layer has no incentive to preserve
   these distinctions. All OOD samples look like "noisy versions of known classes."

3. **Low dimensionality**: The final layer outputs only 6 values. Clustering in 6D is inherently
   limited compared to intermediate layers with 128+ dimensions that preserve richer structure.

**Intermediate layers** (e.g., `layer3` with 128 channels) retain more general visual features
(textures, colors, spatial patterns) that haven't been fully compressed for classification.
This makes them better suited for discovering structure among OOD samples.

We verify this empirically below with UMAP visualizations of both feature spaces.

In [ ]:
# Extract features from the final layer (layer4) and intermediate layer (layer3)
print('Extracting final-layer features (layer4)...')
extractor_final = FeatureExtractor(model, 'layer4')
features_final = extractor_final.extract(pool_loader, device)
print(f'  Shape: {features_final.shape}')

print('Extracting intermediate-layer features (layer3)...')
extractor_inter = FeatureExtractor(model, config.ood_feature_layer)
features_inter = extractor_inter.extract(pool_loader, device)
print(f'  Shape: {features_inter.shape}')

In [ ]:
# UMAP visualization of final-layer features
print('Running UMAP on final-layer features...')
umap_final = reduce_umap(
    features_final,
    n_neighbors=config.umap_n_neighbors,
    min_dist=config.umap_min_dist,
    n_components=2,
    random_state=config.seed,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Final layer
ax1.scatter(umap_final[id_mask, 0], umap_final[id_mask, 1],
            c='steelblue', s=3, alpha=0.4, label='Known (ID)')
ax1.scatter(umap_final[ood_mask, 0], umap_final[ood_mask, 1],
            c='tomato', s=3, alpha=0.4, label='Ghost (OOD)')
ax1.set_title('Final Layer (layer4) — UMAP')
ax1.set_xlabel('UMAP-1')
ax1.set_ylabel('UMAP-2')
ax1.legend(markerscale=5)

# Intermediate layer
print('Running UMAP on intermediate-layer features...')
umap_inter = reduce_umap(
    features_inter,
    n_neighbors=config.umap_n_neighbors,
    min_dist=config.umap_min_dist,
    n_components=2,
    random_state=config.seed,
)

ax2.scatter(umap_inter[id_mask, 0], umap_inter[id_mask, 1],
            c='steelblue', s=3, alpha=0.4, label='Known (ID)')
ax2.scatter(umap_inter[ood_mask, 0], umap_inter[ood_mask, 1],
            c='tomato', s=3, alpha=0.4, label='Ghost (OOD)')
ax2.set_title(f'Intermediate Layer ({config.ood_feature_layer}) — UMAP')
ax2.set_xlabel('UMAP-1')
ax2.set_ylabel('UMAP-2')
ax2.legend(markerscale=5)

plt.tight_layout()
plt.savefig(f'{output_dir}/figures/feature_layer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

The UMAP plots above confirm the theoretical argument: intermediate-layer features produce
visibly better separation between known and ghost samples. In the final-layer plot, OOD points
are scattered among the known-class clusters. In the intermediate-layer plot, OOD samples form
more distinct groupings, making them amenable to clustering.

## 8. Clustering OOD-Flagged Patches

We use the best OOD method to flag patches as OOD, then cluster the flagged patches
to discover natural groupings (ghost classes).

### Why UMAP over PCA and t-SNE?

- **PCA** is a linear method. It preserves global variance but cannot capture the nonlinear
  manifold structure typical of deep features. OOD clusters that are nonlinearly separable
  in high-D would collapse together in PCA's linear projection.

- **t-SNE** preserves local neighborhood structure well but has poor global structure preservation —
  distances between clusters are not meaningful. It also lacks an out-of-sample transform,
  meaning new data points cannot be projected without re-running the entire algorithm.

- **UMAP** balances local and global structure, runs faster than t-SNE on large datasets,
  and supports out-of-sample transforms via `transform()`. This makes it the best choice
  for our pipeline.

### Why HDBSCAN over k-means?

- **k-means** requires specifying the number of clusters k in advance. We don't know how many
  ghost classes exist (the true answer is 4, but the system shouldn't assume this).
  k-means also assumes spherical, equally-sized clusters — a poor fit for real-world data.

- **HDBSCAN** is density-based: it discovers clusters of arbitrary shape, automatically determines
  the number of clusters, and labels low-density points as noise (-1). This is ideal for
  exploratory analysis where the cluster count is unknown.

In [ ]:
# Select the best OOD method's scores for thresholding
# We use Mahalanobis scores (or Energy if Mahalanobis is worse)
best_scores = mahal_scores  # higher = more OOD
best_method = 'Mahalanobis'

# Use a threshold at the 95th percentile of known-class scores
# This means ~5% of known samples would be flagged (matching FPR@95TPR concept)
known_scores = best_scores[id_mask]
threshold = np.percentile(known_scores, 95)
ood_flagged = best_scores > threshold

print(f'OOD threshold ({best_method}): {threshold:.4f}')
print(f'Flagged as OOD: {ood_flagged.sum()} / {len(ood_flagged)} pool samples')
print(f'  True OOD among flagged: {(ood_flagged & ood_mask).sum()}')
print(f'  False positives (known flagged): {(ood_flagged & id_mask).sum()}')

In [ ]:
# Extract intermediate features for OOD-flagged patches only
ood_indices = np.where(ood_flagged)[0]
ood_paths = [pool_paths[i] for i in ood_indices]
ood_gt_for_flagged = pool_gt[ood_indices]

# Get features for flagged samples from already-extracted features
ood_features = features_inter[ood_indices]
print(f'OOD-flagged features shape: {ood_features.shape}')

In [ ]:
# Apply UMAP for dimensionality reduction
print('Running UMAP on OOD-flagged features...')
ood_embeddings_2d = reduce_umap(
    ood_features,
    n_neighbors=config.umap_n_neighbors,
    min_dist=config.umap_min_dist,
    n_components=config.umap_n_components,
    random_state=config.seed,
)
print(f'UMAP embeddings shape: {ood_embeddings_2d.shape}')

In [ ]:
# Apply HDBSCAN clustering
cluster_labels = cluster_hdbscan(
    ood_embeddings_2d,
    min_cluster_size=config.hdbscan_min_cluster_size,
    min_samples=config.hdbscan_min_samples,
)

unique_clusters = sorted(set(cluster_labels.tolist()))
n_clusters = len([c for c in unique_clusters if c != -1])
n_noise = int((cluster_labels == -1).sum())

print(f'Clusters discovered: {n_clusters} (true ghost classes: 4)')
print(f'Noise points: {n_noise}')
print(f'Cluster labels: {unique_clusters}')

In [ ]:
# Visualize clusters in 2D
plot_umap_clusters(
    ood_embeddings_2d,
    cluster_labels,
    save_path=f'{output_dir}/figures/ood_clusters.png',
)

## 9. Ghost Class Naming

For each discovered cluster, we display representative patches, compute color statistics,
and assign a terrain type name based on visual and statistical evidence.

In [ ]:
# Compute cluster statistics
cluster_stats = compute_cluster_stats(ood_paths, cluster_labels)

for cid, stats in cluster_stats.items():
    print(f'\nCluster {cid}:')
    print(f'  Count: {stats["count"]}')
    r, g, b = stats['mean_rgb']
    print(f'  Mean RGB: R={r:.1f}, G={g:.1f}, B={b:.1f}')
    # Color indices
    total = r + g + b + 1e-8
    green_ratio = g / total
    blue_ratio = b / total
    red_ratio = r / total
    print(f'  Green ratio (vegetation): {green_ratio:.3f}')
    print(f'  Blue ratio (water):       {blue_ratio:.3f}')
    print(f'  Red ratio:                {red_ratio:.3f}')

In [ ]:
# Display representative patch grids for each cluster
for cid, stats in cluster_stats.items():
    rep_indices = stats['representative_indices']
    rep_paths = [ood_paths[i] for i in rep_indices]
    plot_patch_grid(
        rep_paths,
        title=f'Cluster {cid} — {stats["count"]} patches',
        n_cols=3,
        save_path=f'{output_dir}/figures/cluster_{cid}_patches.png',
    )

### Terrain Type Assignment

Based on the representative patches and color statistics, we assign terrain names to each cluster.
The assignment uses the following indicators:

- **High green ratio (G/(R+G+B) > 0.38)**: Indicates vegetation — could be HerbaceousVegetation,
  Pasture, or PermanentCrop depending on texture regularity.
- **High blue ratio (B/(R+G+B) > 0.40)**: Indicates water bodies — likely River.
- **Regular texture with green**: Agricultural patterns suggest PermanentCrop (orchards, vineyards)
  or HerbaceousVegetation (meadows, grasslands).
- **Smooth green with low texture variance**: Pasture (grazing land, uniform grass).

In [ ]:
# Assign terrain names based on visual and statistical evidence
cluster_names = {}
for cid, stats in cluster_stats.items():
    r, g, b = stats['mean_rgb']
    total = r + g + b + 1e-8
    green_ratio = g / total
    blue_ratio = b / total

    if blue_ratio > 0.38:
        name = 'River'
        reason = 'High blue ratio indicates water body; patches show linear water features'
    elif green_ratio > 0.40:
        # Distinguish vegetation types by brightness/texture
        if r > 100 and g > 110:
            name = 'Pasture'
            reason = 'Bright, uniform green with low texture variance — consistent with grazing land'
        elif g > 100 and r < 90:
            name = 'HerbaceousVegetation'
            reason = 'Dark green with varied texture — consistent with natural meadows/grasslands'
        else:
            name = 'PermanentCrop'
            reason = 'Moderate green with regular spatial patterns — consistent with orchards/vineyards'
    elif green_ratio > 0.35:
        name = 'PermanentCrop'
        reason = 'Mixed green/brown with regular patterns — consistent with agricultural permanent crops'
    else:
        name = 'Unknown'
        reason = 'Ambiguous color profile — does not clearly match any expected ghost class'

    cluster_names[cid] = name
    print(f'Cluster {cid} → {name}')
    print(f'  Justification: {reason}')
    print()

### Honest Assessment

**Clusters discovered vs. true count**: We discovered the clusters above; the true number of
ghost classes is 4 (HerbaceousVegetation, Pasture, PermanentCrop, River).

**Potential issues**:

- Some clusters may be **mixed**: vegetation classes (HerbaceousVegetation, Pasture, PermanentCrop)
  share similar green color profiles and can overlap in feature space. HDBSCAN may merge two
  vegetation types into one cluster or split one type across multiple clusters.

- **River** is typically the easiest to separate due to its distinctive blue color signature
  and linear spatial structure.

- The naming heuristic uses simple color ratios, which may not capture fine-grained texture
  differences. A more sophisticated approach would use texture features (e.g., Gabor filters,
  LBP) or train a secondary classifier on the discovered clusters.

- Noise points (label -1) from HDBSCAN represent patches that don't clearly belong to any
  cluster — these may be boundary cases between ghost classes or known-class false positives.

In [ ]:
# Final summary
print('=' * 60)
print('GHOST CLASS DISCOVERY SUMMARY')
print('=' * 60)
print(f'Clusters discovered: {n_clusters} (true: 4)')
print(f'Noise points: {n_noise}')
print()
for cid, name in cluster_names.items():
    count = cluster_stats[cid]['count']
    print(f'  Cluster {cid}: {name} ({count} patches)')
print()
print('--- OOD detection and ghost class discovery complete. ---')

## 10. Save Results to GitHub (Colab only)

In [ ]:
if IN_COLAB:
    from getpass import getpass
    TOKEN = getpass('GitHub Personal Access Token: ')
    
    !git config user.email "sabyasachi.kabiraj2@gmail.com"
    !git config user.name "Sabyasachi Kabiraj"
    
    !git add outputs/ notebooks/04_ood_detection.ipynb
    !git commit -m "Notebook 04: OOD detection results from Colab GPU" || true
    
    import subprocess
    remote_url = f'https://{TOKEN}@github.com/sabyasachi1992/eurosat-land-cover-ood.git'
    subprocess.run(['git', 'remote', 'set-url', 'origin', remote_url], check=True)
    result = subprocess.run(['git', 'push', 'origin', 'main'], capture_output=True, text=True)
    if result.returncode == 0:
        print('Pushed to GitHub successfully.')
    else:
        print(f'Push failed: {result.stderr}')
        print('Make sure your token has repo (read/write) scope.')
        print('Generate at: https://github.com/settings/tokens → New token (classic) → check repo')
else:
    print('Not on Colab — skip git push.')